# БӨЖ 2 — Big Data аналитикасында машиналық оқыту әдістерін қолдану

**Жоба:** Customer Churn Prediction with Apache Spark MLlib  
**Мақсат:** Logistic Regression, Random Forest және Gradient Boosted Trees модельдерін Spark DataFrame / ML Pipeline арқылы салыстыру.


In [ ]:
# Google Colab үшін
!pip -q install pyspark

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt

spark = SparkSession.builder.appName("BigDataMLChurn").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
# 1. Big Data синтетикалық датасетін Spark ішінде генерациялау
ROWS = 200_000   # 1_000_000 немесе одан да көп мәнге өзгертуге болады
SEED = 42

df = spark.range(ROWS).withColumnRenamed("id", "customer_id")
df = (
    df
    .withColumn("age", (F.floor(F.rand(SEED) * 57) + 18).cast("int"))
    .withColumn("tenure_months", (F.floor(F.rand(SEED + 1) * 72) + 1).cast("int"))
    .withColumn("monthly_spend", F.round(F.greatest(F.lit(5.0), F.least(F.lit(180.0), 55 + 22 * F.randn(SEED + 2))), 2))
    .withColumn("usage_hours", F.round(F.greatest(F.lit(1.0), F.least(F.lit(120.0), 42 + 18 * F.randn(SEED + 3))), 2))
    .withColumn("support_calls", F.floor(F.rand(SEED + 4) * 6).cast("int"))
    .withColumn("late_payments", F.floor(F.rand(SEED + 5) * 5).cast("int"))
    .withColumn("satisfaction_score", F.round(1 + 4 * F.rand(SEED + 6), 2))
    .withColumn("contract_type",
        F.when(F.rand(SEED + 7) < 0.55, "month-to-month")
         .when(F.rand(SEED + 7) < 0.83, "one-year")
         .otherwise("two-year"))
    .withColumn("region",
        F.when(F.rand(SEED + 8) < 0.32, "Almaty")
         .when(F.rand(SEED + 8) < 0.57, "Astana")
         .when(F.rand(SEED + 8) < 0.73, "Shymkent")
         .otherwise("Other"))
    .withColumn("has_internet", F.when(F.rand(SEED + 9) < 0.88, 1).otherwise(0))
)

risk = (
    F.lit(-1.5)
    + 0.55 * F.when(F.col("contract_type") == "month-to-month", 1.0).otherwise(0.0)
    - 0.55 * F.when(F.col("contract_type") == "two-year", 1.0).otherwise(0.0)
    - 0.022 * F.col("tenure_months")
    + 0.18 * F.col("support_calls")
    + 0.28 * F.col("late_payments")
    - 0.75 * (F.col("satisfaction_score") - 3.0)
    + 0.009 * (F.col("monthly_spend") - 55.0)
    - 0.015 * (F.col("usage_hours") - 42.0)
    + 0.25 * (1 - F.col("has_internet"))
    + 0.90 * F.when((F.col("support_calls") >= 3) & (F.col("satisfaction_score") < 3), 1.0).otherwise(0.0)
    + 0.70 * F.when((F.col("monthly_spend") > 80) & (F.col("contract_type") == "month-to-month"), 1.0).otherwise(0.0)
    + 0.65 * F.when(F.col("late_payments") >= 2, 1.0).otherwise(0.0)
    + 0.35 * F.randn(SEED + 10)
)
prob = 1.0 / (1.0 + F.exp(-risk))
df = df.withColumn("label", F.when(F.rand(SEED + 11) < prob, 1.0).otherwise(0.0)).cache()

print("Жазба саны:", f"{df.count():,}")
df.groupBy("label").count().orderBy("label").show()
df.show(5)

In [ ]:
# 2. Preprocessing + train/test
categorical = ["contract_type", "region"]
numeric = ["age", "tenure_months", "monthly_spend", "usage_hours",
           "support_calls", "late_payments", "satisfaction_score", "has_internet"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical],
                        outputCols=[f"{c}_ohe" for c in categorical])
assembler = VectorAssembler(inputCols=numeric + [f"{c}_ohe" for c in categorical],
                            outputCol="features", handleInvalid="keep")

train, test = df.randomSplit([0.8, 0.2], seed=SEED)
counts = {float(r["label"]): int(r["count"]) for r in train.groupBy("label").count().collect()}
total = sum(counts.values())
w0 = total / (2.0 * counts.get(0.0, 1))
w1 = total / (2.0 * counts.get(1.0, 1))
train = train.withColumn("class_weight", F.when(F.col("label") == 1.0, w1).otherwise(w0))

print("Train:", train.count(), "Test:", test.count())

In [ ]:
# 3. Үш машиналық оқыту моделін үйрету және бағалау
models = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="label", weightCol="class_weight", maxIter=80, regParam=0.02),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="class_weight", numTrees=100, maxDepth=10, seed=SEED),
    "Gradient Boosted Trees": GBTClassifier(featuresCol="features", labelCol="label", weightCol="class_weight", maxIter=60, maxDepth=6, stepSize=0.08, seed=SEED),
}

auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

def positive_metrics(pred):
    r = pred.select(
        F.sum(F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
        F.sum(F.when((F.col("label") == 0) & (F.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("label") == 1) & (F.col("prediction") == 0), 1).otherwise(0)).alias("fn"),
    ).first()
    tp, fp, fn = float(r.tp), float(r.fp), float(r.fn)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

results = []
for name, estimator in models.items():
    pipeline = Pipeline(stages=indexers + [encoder, assembler, estimator])
    fitted = pipeline.fit(train)
    pred = fitted.transform(test).cache()
    precision, recall, f1 = positive_metrics(pred)
    results.append({
        "model": name,
        "accuracy": acc_eval.evaluate(pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc_eval.evaluate(pred),
    })
    pred.unpersist()

results_df = pd.DataFrame(results).sort_values("auc", ascending=False)
results_df

In [ ]:
# 4. Модельдерді графикпен салыстыру
ax = results_df.set_index("model")[["accuracy", "f1", "auc"]].plot(kind="bar", figsize=(10, 5))
ax.set_ylim(0, 1)
ax.set_ylabel("Метрика мәні")
ax.set_xlabel("")
ax.set_title("Spark ML модельдерін салыстыру")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Қорытынды

- Spark ML Pipeline деректерді дайындау мен модельді үйретуді бір workflow-ға біріктіреді.
- `ROWS` параметрін көбейту арқылы есепті үлкен деректерге масштабтауға болады.
- Модельдер Accuracy, Precision, Recall, F1 және ROC-AUC метрикаларымен салыстырылады.
- Gradient Boosted Trees күрделі сызықтық емес тәуелділіктерді ұстауға ыңғайлы, ал Logistic Regression baseline ретінде түсіндіруге жеңіл.
